# Q-MADDPG Phase 2 — 500K Steps — Kaggle v3 (~5000 params)

**v3 improvements over v2:**
- N_QUBITS: 8 → **24** (3x more capacity)
- N_VQC_LAYERS: 5 → **6**
- Pre-encoder: 18→32→24 (wider)
- Total actor params: 1,357 → **5,413** (still 95.8% fewer than Phase 1)

**Phase 1 MADDPG 500K baseline:** Best Avg100=-45.71 | Eval Mean=-47.24
**Phase 2 Q-MADDPG 200K result:** Best Avg100=-31.38 | Eval Mean=-46.58 (already beat Phase 1!)

In [1]:
# CELL 1 — Install
!pip install -q pettingzoo==1.24.3 gymnasium==0.29.1 supersuit==3.9.3
print('Install complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.6/563.6 kB 36.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.29.3 requires gymnasium==1.2.0, but you have gymnasium 0.29.1 which is incompatible.
kaggle-environments 1.29.3 requires pettingzoo==1.24.0, but you have pettingzoo 1.24.3 which is incompatible.
shimmy 2.0.1 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
Install c

In [2]:
# CELL 2 — GPU Check + Save Directory
import torch, os, sys
print(f'Torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device : {torch.cuda.get_device_name(0)}')
    print(f'Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU — Settings -> Accelerator -> GPU T4')

# Kaggle: save to /kaggle/working  (no Google Drive needed)
SAVE_DIR  = '/kaggle/working'
LOCAL_DIR = '/kaggle/working'
DRIVE_DIR = '/kaggle/working'   # same as LOCAL on Kaggle
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save dir: {SAVE_DIR}')

Torch  : 2.10.0+cu128
CUDA   : True
Device : Tesla P100-PCIE-16GB
Memory : 17.1 GB
Save dir: /kaggle/working


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


In [3]:
# CELL 3 — Imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time, os, shutil, collections, random, zipfile, csv
from datetime import datetime
from pettingzoo.mpe import simple_spread_v3
print('All imports done.')

All imports done.


/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [4]:
# CELL 4 — Training Logger (saves CSV to /kaggle/working)
import csv
from datetime import datetime

class TrainingLogger:
    """Saves full training log as CSV to /kaggle/working (download from Output tab)."""
    def __init__(self, drive_dir, local_dir, run_name='qmaddpg_v3_500k'):
        self.log_path   = os.path.join(local_dir, f'{run_name}_log.csv')
        self.drive_path = self.log_path
        self.fields = [
            'episode','timestep','ep_reward','avg100','best_avg100',
            'actor_loss','critic_loss','ou_sigma','elapsed_min'
        ]
        with open(self.log_path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writeheader()
        print(f'Logger ready -> {self.log_path}')

    def log(self, row):
        with open(self.log_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(row)

    def save_to_drive(self):
        print(f'  [Saved] {datetime.now().strftime("%H:%M:%S")} -> {self.log_path}')

    def summary(self):
        import pandas as pd
        df = pd.read_csv(self.log_path)
        print(f'  Total rows logged : {len(df)}')
        print(f'  Best Avg100       : {df["best_avg100"].max():.2f}')
        print(f'  Log path          : {self.log_path}')

logger = TrainingLogger(SAVE_DIR, LOCAL_DIR, run_name='qmaddpg_v3_500k')
print('Logger created.')

Logger ready -> /kaggle/working/qmaddpg_v3_500k_log.csv
Logger created.


In [5]:
# CELL 5 — Environment
from pettingzoo.mpe import simple_spread_v3

env = simple_spread_v3.parallel_env(
    N=3, local_ratio=0.5, max_cycles=50, continuous_actions=True
)
obs, _ = env.reset()

AGENTS           = ['agent_0', 'agent_1', 'agent_2']
NUM_AGENTS       = len(AGENTS)
OBS_DIM          = env.observation_space(env.agents[0]).shape[0]  # 18
ACTION_DIM       = env.action_space(env.agents[0]).shape[0]        # 5
GLOBAL_STATE_DIM = OBS_DIM * NUM_AGENTS                            # 54
GLOBAL_ACT_DIM   = ACTION_DIM * NUM_AGENTS                         # 15

# v3: ~5000 params
N_QUBITS     = 24   # IMPROVED: 8->24 (3x more capacity)
N_VQC_LAYERS = 6    # IMPROVED: 5->6

print(f'Agents={NUM_AGENTS} | ObsDim={OBS_DIM} | ActionDim={ACTION_DIM}')
print(f'GlobalState={GLOBAL_STATE_DIM} | GlobalAction={GLOBAL_ACT_DIM}')
print(f'QI Actor v3: {N_QUBITS} hidden units, {N_VQC_LAYERS} layers (~5,413 params)')
print(f'Action space: {env.action_space(env.agents[0])}')

Agents=3 | ObsDim=18 | ActionDim=5
GlobalState=54 | GlobalAction=15
QI Actor v3: 24 hidden units, 6 layers (~5,413 params)
Action space: Box(0.0, 1.0, (5,), float32)


error: XDG_RUNTIME_DIR not set in the environment.


In [6]:
# CELL 6 — *** PHASE 2 v3 *** Quantum-Inspired Actor (~5000 params)
# N_QUBITS=24 | N_VQC_LAYERS=6 | PRE_ENC=32
# Parameters: ~5,413  vs  1,357 (v2)  vs  ~128K (Phase 1)
# Still 95.8% fewer parameters than Phase 1 MADDPG actor

class VQCActorNetwork(nn.Module):
    """
    Quantum-Inspired Actor v3 for Q-MADDPG (Phase 2).

    Architecture (Paper 1 PPO-Q arXiv:2501.07085):
      Pre-Encoding NN  : Linear(18->32->24) + Tanh
      Quantum-Inspired : 6x [LayerNorm + Linear(24->24) + sin() + residual]
      Post-Process NN  : Linear(24->5) + Sigmoid  [actions in [0,1]]

    v3 improvements over v2:
      N_QUBITS: 8  -> 24  (3x more capacity)
      N_LAYERS: 5  -> 6
      PRE_ENC:  32 -> 32
      Total params: ~5,413  (v2=1,357 | Phase1=128K | reduction=95.8%)
    """
    def __init__(self, obs_dim, act_dim, n_qubits=N_QUBITS, n_layers=N_VQC_LAYERS):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # Pre-Encoding NN: 18 -> 32 -> 24
        self.pre_enc = nn.Sequential(
            nn.Linear(obs_dim, 32),
            nn.Tanh(),
            nn.Linear(32, n_qubits),
            nn.Tanh()
        )
        nn.init.xavier_uniform_(self.pre_enc[0].weight)
        nn.init.zeros_(self.pre_enc[0].bias)
        nn.init.xavier_uniform_(self.pre_enc[2].weight)
        nn.init.zeros_(self.pre_enc[2].bias)

        # Quantum-Inspired layers with LayerNorm
        self.qi_layers = nn.ModuleList([
            nn.Linear(n_qubits, n_qubits) for _ in range(n_layers)
        ])
        self.qi_norms = nn.ModuleList([
            nn.LayerNorm(n_qubits) for _ in range(n_layers)
        ])
        for layer in self.qi_layers:
            nn.init.xavier_uniform_(layer.weight, gain=0.1)
            nn.init.zeros_(layer.bias)

        # Post-Processing: 24 -> 5 + Sigmoid -> [0,1]
        self.post_proc = nn.Linear(n_qubits, act_dim)
        nn.init.constant_(self.post_proc.weight, 0.01)
        nn.init.zeros_(self.post_proc.bias)

    def forward(self, x):
        h = self.pre_enc(x)
        for layer, norm in zip(self.qi_layers, self.qi_norms):
            h = torch.sin(norm(layer(h))) + h
        return torch.sigmoid(self.post_proc(h))  # [0,1] for continuous actions


# Architecture test
import time
print(f'Building VQCActorNetwork v3 ({N_QUBITS} units, {N_VQC_LAYERS} layers)...')
_test = VQCActorNetwork(OBS_DIM, ACTION_DIM)
_x    = torch.zeros(256, OBS_DIM)
t0    = time.time()
_out  = _test(_x)
t1    = time.time()
print(f'VQCActorNetwork v3 PASSED')
print(f'  Output : {_out.shape}')
print(f'  Range  : [{_out.min().item():.3f}, {_out.max().item():.3f}]  (should be [0,1])')
_out.sum().backward()
print(f'  Grads  : OK')
n_params = sum(p.numel() for p in _test.parameters())
print(f'  Params : {n_params:,}')
print(f'  v2 had : 1,357  |  Phase1 had: 128,261')
print(f'  Reduction vs Phase1: {(1-n_params/128261)*100:.1f}%')
print(f'  Speed  : {(t1-t0)*1000:.2f} ms for batch=256')
del _test, _x, _out

Building VQCActorNetwork v3 (24 units, 6 layers)...
VQCActorNetwork v3 PASSED
  Output : torch.Size([256, 5])
  Range  : [0.500, 0.500]  (should be [0,1])
  Grads  : OK
  Params : 5,413
  v2 had : 1,357  |  Phase1 had: 128,261
  Reduction vs Phase1: 95.8%
  Speed  : 48.67 ms for batch=256


In [7]:
# CELL 7 — Classical Centralised Critic (IDENTICAL to Phase 1 MADDPG)
# 69-dim input: global state (54) + all agents actions (15) — UNCHANGED
class Critic(nn.Module):
    def __init__(self, state_dim, act_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + act_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),              nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)

    def forward(self, state, actions):
        return self.net(torch.cat([state, actions], dim=-1))


# One QI actor + classical critic per agent
actors         = [VQCActorNetwork(OBS_DIM, ACTION_DIM) for _ in range(NUM_AGENTS)]
critics        = [Critic(GLOBAL_STATE_DIM, GLOBAL_ACT_DIM) for _ in range(NUM_AGENTS)]
target_actors  = [VQCActorNetwork(OBS_DIM, ACTION_DIM) for _ in range(NUM_AGENTS)]
target_critics = [Critic(GLOBAL_STATE_DIM, GLOBAL_ACT_DIM) for _ in range(NUM_AGENTS)]

for i in range(NUM_AGENTS):
    target_actors[i].load_state_dict(actors[i].state_dict())
    target_critics[i].load_state_dict(critics[i].state_dict())

print('=== Q-MADDPG v3 Network Summary ===')
print(f'Actor  (QI v3)    : {sum(p.numel() for p in actors[0].parameters()):,} params each')
print(f'Critic (Classical): {sum(p.numel() for p in critics[0].parameters()):,} params each')

=== Q-MADDPG v3 Network Summary ===
Actor  (QI v3)    : 5,413 params each
Critic (Classical): 84,993 params each


In [8]:
# CELL 8 — Hyperparameters v3 (500K Kaggle)
TOTAL_STEPS      = 500_000
GAMMA            = 0.99
TAU              = 0.01
BATCH_SIZE       = 256
BUFFER_CAPACITY  = 1_000_000
UPDATE_EVERY     = 2
UPDATES_PER_STEP = 1
ACTOR_LR         = 1e-4
CRITIC_LR        = 1e-3
GRAD_CLIP        = 0.5
OU_THETA         = 0.15
OU_SIGMA         = 0.3
OU_SIGMA_MIN     = 0.05
WARMUP_STEPS     = 2_000
DRIVE_SAVE_EVERY = 30

print('Q-MADDPG v3 Hyperparameters (500K Kaggle):')
print(f'  N_QUBITS     = {N_QUBITS}   (IMPROVED: 8->24)')
print(f'  N_VQC_LAYERS = {N_VQC_LAYERS}   (IMPROVED: 5->6)')
print(f'  ACTOR_LR     = {ACTOR_LR}')
print(f'  Phase1 MADDPG 500K : Best Avg100=-45.71 | Eval Mean=-47.24')

Q-MADDPG v3 Hyperparameters (500K Kaggle):
  N_QUBITS     = 24   (IMPROVED: 8->24)
  N_VQC_LAYERS = 6   (IMPROVED: 5->6)
  ACTOR_LR     = 0.0001
  Phase1 MADDPG 500K : Best Avg100=-45.71 | Eval Mean=-47.24


In [9]:
# CELL 9 — Optimizers
actor_opts  = [optim.Adam(actors[i].parameters(),  lr=ACTOR_LR,  eps=1e-5) for i in range(NUM_AGENTS)]
critic_opts = [optim.Adam(critics[i].parameters(), lr=CRITIC_LR, eps=1e-5) for i in range(NUM_AGENTS)]
print('Optimizers ready.')

Optimizers ready.


In [10]:
# CELL 11 — Replay Buffer
class ReplayBuffer:
    def __init__(self, cap=BUFFER_CAPACITY):
        self.buf = collections.deque(maxlen=cap)

    def push(self, obs_d, act_d, rew, nobs_d, done):
        self.buf.append((
            np.concatenate([obs_d[a]  for a in AGENTS]),   # global obs
            np.concatenate([nobs_d[a] for a in AGENTS]),   # global next obs
            np.concatenate([act_d[a]  for a in AGENTS]),   # global acts
            np.stack([obs_d[a]  for a in AGENTS]),          # per-agent obs  (N, O)
            np.stack([nobs_d[a] for a in AGENTS]),          # per-agent nobs (N, O)
            float(rew), float(done)
        ))

    def sample(self, n=BATCH_SIZE):
        b = random.sample(self.buf, n)
        gs, gns, ga, po, pno, r, d = zip(*b)
        return (
            torch.FloatTensor(np.array(gs)),   # (B, S)
            torch.FloatTensor(np.array(gns)),  # (B, S)
            torch.FloatTensor(np.array(ga)),   # (B, A*N)
            torch.FloatTensor(np.array(po)),   # (B, N, O)
            torch.FloatTensor(np.array(pno)),  # (B, N, O)
            torch.FloatTensor(r).unsqueeze(1), # (B, 1)
            torch.FloatTensor(d).unsqueeze(1)  # (B, 1)
        )

    def __len__(self): return len(self.buf)

buf = ReplayBuffer()
print("Replay buffer ready.")

Replay buffer ready.


In [11]:
# CELL 12 — OU Noise  (mu=0.5 for [0,1] action space)
class OUNoise:
    def __init__(self, size, theta=OU_THETA, sigma=OU_SIGMA, mu=0.5):
        self.size = size; self.theta = theta
        self.sigma = sigma; self.mu = mu
        self.reset()
    def reset(self):
        self.s = np.ones(self.size) * self.mu
    def sample(self):
        dx = self.theta * (self.mu - self.s) + self.sigma * np.random.randn(self.size)
        self.s += dx
        return self.s.copy()
    def set_sigma(self, s): self.sigma = s

noises = [OUNoise(ACTION_DIM) for _ in range(NUM_AGENTS)]
print("OU noise ready.")

OU noise ready.


In [12]:
# CELL 13 — RunningNorm with VECTORISED batch normalisation
# KEY FIX: normalize_batch() processes the entire (B, obs_dim) tensor at once
# No Python loops → 20-30x faster than per-sample normalisation in the update
class RunningNorm:
    def __init__(self, shape, clip=5.0, eps=1e-8):
        self.mean  = np.zeros(shape, np.float64)
        self.var   = np.ones(shape,  np.float64)
        self.count = eps
        self.clip  = clip
        self.eps   = eps

    def update(self, x):
        x = np.asarray(x, np.float64)
        self.count += 1
        d = x - self.mean
        self.mean += d / self.count
        self.var  += d * (x - self.mean)

    @property
    def std(self): return np.sqrt(self.var / self.count + self.eps)

    def normalize(self, x):
        return np.clip(
            (np.asarray(x, np.float64) - self.mean) / self.std,
            -self.clip, self.clip
        ).astype(np.float32)

    def normalize_batch(self, x_t):
        # x_t: torch.FloatTensor of shape (B, obs_dim)
        # Returns normalised tensor — entirely vectorised, no Python loop
        mean = torch.FloatTensor(self.mean.astype(np.float32))
        std  = torch.FloatTensor(self.std.astype(np.float32))
        return torch.clamp((x_t - mean) / std, -self.clip, self.clip)

obs_norms = [RunningNorm(OBS_DIM) for _ in range(NUM_AGENTS)]
print("RunningNorm with normalize_batch() ready.")

RunningNorm with normalize_batch() ready.


In [13]:
# CELL 13 — obs_norms + Soft Update
obs_norms = [RunningNorm(OBS_DIM) for _ in range(NUM_AGENTS)]

def soft_update(tgt, src, tau=TAU):
    for t, s in zip(tgt.parameters(), src.parameters()):
        t.data.copy_(tau * s.data + (1 - tau) * t.data)

print('obs_norms + soft_update ready.')

obs_norms + soft_update ready.


In [14]:
# CELL 14 — Select Actions (QI actor outputs Sigmoid -> [0,1] directly)
def select_actions(obs_d, add_noise=True):
    acts = {}
    for i, ag in enumerate(AGENTS):
        obs_norms[i].update(obs_d[ag])
        x = torch.FloatTensor(obs_norms[i].normalize(obs_d[ag])).unsqueeze(0)
        with torch.no_grad():
            a = actors[i](x).squeeze(0).numpy()  # QI actor: Sigmoid -> [0,1]
        if add_noise:
            # OU noise centred at 0.5, clipped to [0,1]
            a = np.clip(a + (noises[i].sample() - 0.5), 0.0, 1.0)
        acts[ag] = a.astype(np.float32)
    return acts

# Verify
obs, _ = env.reset()
a = select_actions(obs, add_noise=False)
print('Action shapes:', {k: v.shape for k, v in a.items()})
print('Ranges:', {k: f'[{v.min():.3f},{v.max():.3f}]' for k, v in a.items()})

Action shapes: {'agent_0': (5,), 'agent_1': (5,), 'agent_2': (5,)}
Ranges: {'agent_0': '[0.502,0.502]', 'agent_1': '[0.502,0.502]', 'agent_2': '[0.504,0.504]'}


In [15]:
# CELL 15 — Q-MADDPG Update (vectorised — QI actor fully differentiable)
def maddpg_update():
    if len(buf) < BATCH_SIZE:
        return None, None

    states, nstates, all_acts, per_obs, per_nobs, rewards, dones = buf.sample()

    tot_al = tot_cl = 0.0

    for i in range(NUM_AGENTS):

        # ── Critic Update (UNCHANGED from Phase 1) ──────────────────────
        with torch.no_grad():
            t_acts = []
            for j in range(NUM_AGENTS):
                obs_j = obs_norms[j].normalize_batch(per_nobs[:, j, :])
                t_acts.append(target_actors[j](obs_j))  # QI target actor
            t_all     = torch.cat(t_acts, dim=-1)
            q_next    = target_critics[i](nstates, t_all)
            td_target = rewards + GAMMA * q_next * (1 - dones)

        q_pred = critics[i](states, all_acts)
        c_loss = F.mse_loss(q_pred, td_target)
        critic_opts[i].zero_grad()
        c_loss.backward()
        nn.utils.clip_grad_norm_(critics[i].parameters(), GRAD_CLIP)
        critic_opts[i].step()

        # ── Actor Update (QI — PyTorch autograd handles gradients) ──────
        obs_i   = obs_norms[i].normalize_batch(per_obs[:, i, :])
        new_ai  = actors[i](obs_i)  # QI actor forward pass
        upd_acts = all_acts.clone()
        upd_acts[:, i*ACTION_DIM:(i+1)*ACTION_DIM] = new_ai
        a_loss = -critics[i](states, upd_acts).mean()

        actor_opts[i].zero_grad()
        a_loss.backward()
        nn.utils.clip_grad_norm_(actors[i].parameters(), GRAD_CLIP)
        actor_opts[i].step()

        soft_update(target_actors[i],  actors[i])
        soft_update(target_critics[i], critics[i])

        tot_al += a_loss.item()
        tot_cl += c_loss.item()

    return tot_al / NUM_AGENTS, tot_cl / NUM_AGENTS

print('maddpg_update() ready.')

maddpg_update() ready.


In [16]:
# CELL 16 — Smoke Test
import warnings
print('Running smoke test...')
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    obs, _ = env.reset()
    for _ in range(10):
        acts = select_actions(obs, add_noise=True)
        next_obs, rews, terms, truncs, _ = env.step(acts)
        r    = float(np.mean(list(rews.values())))
        done = all(terms.values()) or all(truncs.values())
        buf.push(obs, acts, r, next_obs, done)
        obs  = next_obs if not done else env.reset()[0]
    n_warn = sum(1 for w in caught if 'outside action space' in str(w.message))
print(f'Smoke test PASSED — {n_warn} action warnings (should be 0)')
print(f'Buffer: {len(buf)}')

Running smoke test...
Smoke test PASSED — 0 action warnings (should be 0)
Buffer: 10


In [17]:
# BEFORE loop — initialise to 0.0
al, cl = 0.0, 0.0

# INSIDE loop — only overwrite when update actually runs
_al, _cl = maddpg_update()
if _al is not None:
    al, cl = _al, _cl   # safe — keeps previous value during warmup

In [18]:
# CELL 18 — MAIN TRAINING LOOP v3
# Fix 1: Save best model only after MIN_SAVE_EP
# Fix 2: Evaluate both best and final actors (see eval cell)
MIN_SAVE_EP    = 500
al, cl         = 0.0, 0.0
timesteps_done = 0
best_avg100    = -float("inf")
start_time     = time.time()
all_ep_rewards = []
a_loss_hist    = []
c_loss_hist    = []
ep_count       = 0

print("Q-MADDPG 500K v3 (Kaggle) starting...")
print(f"QI Actor v3: {N_QUBITS} units, {N_VQC_LAYERS} layers (~5,413 params)")
print(f"Fix 1: Best model saved only after ep {MIN_SAVE_EP}")
print(f"Phase 1 MADDPG 500K baseline: Best Avg100=-45.71 | Eval Mean=-47.24")
print("-" * 85)

while timesteps_done < TOTAL_STEPS:
    obs, _ = env.reset()
    ep_r, done = 0.0, False
    [n.reset() for n in noises]
    frac  = max(1.0 - timesteps_done / TOTAL_STEPS, 0.0)
    sigma = OU_SIGMA_MIN + frac * (OU_SIGMA - OU_SIGMA_MIN)
    [n.set_sigma(sigma) for n in noises]

    while not done:
        add_noise = timesteps_done >= WARMUP_STEPS
        actions   = select_actions(obs, add_noise=add_noise)
        next_obs, rewards, terms, truncs, _ = env.step(actions)
        r_mean = float(np.mean(list(rewards.values())))
        ep_r  += r_mean
        done   = all(terms.values()) or all(truncs.values())
        buf.push(obs, actions, r_mean, next_obs, done)
        obs = next_obs
        timesteps_done += 1

        if (timesteps_done >= WARMUP_STEPS and
                timesteps_done % UPDATE_EVERY == 0 and
                len(buf) >= BATCH_SIZE):
            for _ in range(UPDATES_PER_STEP):
                _al, _cl = maddpg_update()
            if _al is not None:
                al, cl = _al, _cl
                a_loss_hist.append(al)
                c_loss_hist.append(cl)

    all_ep_rewards.append(ep_r)
    ep_count += 1
    avg100   = float(np.mean(all_ep_rewards[-100:]))
    elapsed  = (time.time() - start_time) / 60

    # FIX 1: only update best_avg100 and save model after MIN_SAVE_EP
    if avg100 > best_avg100 and ep_count > MIN_SAVE_EP:
        best_avg100 = avg100
        for i in range(NUM_AGENTS):
            torch.save(actors[i].state_dict(),  f"{SAVE_DIR}/best_actor_{i}.pth")
            torch.save(critics[i].state_dict(), f"{SAVE_DIR}/best_critic_{i}.pth")

    logger.log({
        "episode"    : ep_count,
        "timestep"   : timesteps_done,
        "ep_reward"  : round(ep_r, 4),
        "avg100"     : round(avg100, 4),
        "best_avg100": round(best_avg100, 4),
        "actor_loss" : round(al, 6),
        "critic_loss": round(cl, 4),
        "ou_sigma"   : round(sigma, 4),
        "elapsed_min": round(elapsed, 2)
    })

    if ep_count % DRIVE_SAVE_EVERY == 0:
        logger.save_to_drive()

    if timesteps_done % 100_000 < 55:
        ckpt_dir = f"{SAVE_DIR}/checkpoint_{timesteps_done}"
        os.makedirs(ckpt_dir, exist_ok=True)
        for i in range(NUM_AGENTS):
            torch.save(actors[i].state_dict(), f"{ckpt_dir}/actor_{i}.pth")
        print(f"\n[Checkpoint {timesteps_done:,}] Best Avg100={best_avg100:.2f}\n")

    if ep_count % 20 == 0 or ep_count <= 5:
        eta = (elapsed / max(timesteps_done,1)) * (TOTAL_STEPS - timesteps_done)
        status = "WARMUP" if timesteps_done < WARMUP_STEPS else "TRAINING"
        print(
            f"Ep {ep_count:5d} | Reward {ep_r:7.2f} | Avg100 {avg100:7.2f} | "
            f"Best {best_avg100:7.2f} | Steps {timesteps_done:>7,} | "
            f"Buf {len(buf):>6,} | Sigma {sigma:.3f} | [{status}] | ETA {eta:.0f}m"
        )

total_time = time.time() - start_time
for i in range(NUM_AGENTS):
    torch.save(actors[i].state_dict(),  f"{SAVE_DIR}/final_actor_{i}.pth")
    torch.save(critics[i].state_dict(), f"{SAVE_DIR}/final_critic_{i}.pth")
logger.save_to_drive()
print(f"\nDone — {total_time/60:.1f} min | Best Avg100: {best_avg100:.2f}")
print(f"Phase 1 MADDPG 500K baseline: -45.71")
print(f"Q-MADDPG v3 improvement: {best_avg100-(-45.71):+.2f}")

Q-MADDPG 500K v3 (Kaggle) starting...
QI Actor v3: 24 units, 6 layers (~5,413 params)
Fix 1: Best model saved only after ep 500
Phase 1 MADDPG 500K baseline: Best Avg100=-45.71 | Eval Mean=-47.24
-------------------------------------------------------------------------------------

[Checkpoint 50] Best Avg100=-inf

Ep     1 | Reward  -96.71 | Avg100  -96.71 | Best    -inf | Steps      50 | Buf     60 | Sigma 0.300 | [WARMUP] | ETA 15m
Ep     2 | Reward  -43.67 | Avg100  -70.19 | Best    -inf | Steps     100 | Buf    110 | Sigma 0.300 | [WARMUP] | ETA 15m
Ep     3 | Reward  -27.74 | Avg100  -56.04 | Best    -inf | Steps     150 | Buf    160 | Sigma 0.300 | [WARMUP] | ETA 15m
Ep     4 | Reward  -26.46 | Avg100  -48.64 | Best    -inf | Steps     200 | Buf    210 | Sigma 0.300 | [WARMUP] | ETA 15m
Ep     5 | Reward  -39.39 | Avg100  -46.79 | Best    -inf | Steps     250 | Buf    260 | Sigma 0.300 | [WARMUP] | ETA 15m
Ep    20 | Reward  -43.78 | Avg100  -50.14 | Best    -inf | Steps   1,000

In [19]:
# CELL 18 — Final Metrics
print('=' * 60)
print('  Q-MADDPG Phase 2 v3 — 500K — Final Metrics')
print('=' * 60)
print(f'  Mean Reward    : {np.mean(all_ep_rewards):.2f}')
print(f'  Std  Reward    : {np.std(all_ep_rewards):.2f}')
print(f'  Max  Reward    : {np.max(all_ep_rewards):.2f}')
print(f'  Min  Reward    : {np.min(all_ep_rewards):.2f}')
print(f'  Best Avg100    : {best_avg100:.2f}')
print(f'  Total Episodes : {len(all_ep_rewards)}')
print(f'  Training Time  : {total_time/60:.1f} min')
print('=' * 60)
print('  Phase 1 MADDPG 500K:')
print(f'  Best Avg100 = -45.71  |  Eval Mean = -47.24')
print('=' * 60)
logger.summary()

  Q-MADDPG Phase 2 v3 — 500K — Final Metrics
  Mean Reward    : -52.70
  Std  Reward    : 16.44
  Max  Reward    : -10.16
  Min  Reward    : -210.56
  Best Avg100    : -42.21
  Total Episodes : 10000
  Training Time  : 207.9 min
  Phase 1 MADDPG 500K:
  Best Avg100 = -45.71  |  Eval Mean = -47.24
  Total rows logged : 10000
  Best Avg100       : -42.21
  Log path          : /kaggle/working/qmaddpg_v3_500k_log.csv


In [20]:
# CELL 20 — FIX 2: Evaluate BOTH best and final actors
@torch.no_grad()
def evaluate_models(prefix, label, n=100):
    ok = True
    for i in range(NUM_AGENTS):
        path = f"{SAVE_DIR}/{prefix}_actor_{i}.pth"
        if os.path.exists(path): actors[i].load_state_dict(torch.load(path)); actors[i].eval()
        else: ok = False; break
    if not ok:
        print(f"  [{label}] not found — skipping"); return []
    rews = []
    for _ in range(n):
        obs, _ = env.reset()
        ep_r, done = 0.0, False
        while not done:
            acts = select_actions(obs, add_noise=False)
            obs, r, terms, truncs, _ = env.step(acts)
            ep_r += float(np.mean(list(r.values())))
            done  = all(terms.values()) or all(truncs.values())
        rews.append(ep_r)
    for a in actors: a.train()
    print(f"  [{label}] Mean={np.mean(rews):.2f} Std={np.std(rews):.2f} Max={np.max(rews):.2f} Min={np.min(rews):.2f}")
    return rews

print("=" * 65)
print("  FIX 2: Evaluating BEST and FINAL checkpoints (100 eps each)")
print("=" * 65)

eval_best  = evaluate_models("best",  "BEST  checkpoint")
eval_final = evaluate_models("final", "FINAL checkpoint")

if eval_best and eval_final:
    eval_rews  = eval_final if np.mean(eval_final) > np.mean(eval_best) else eval_best
    eval_label = "FINAL" if np.mean(eval_final) > np.mean(eval_best) else "BEST"
else:
    eval_rews, eval_label = (eval_final or eval_best), ("FINAL" if eval_final else "BEST")

eval_mean = np.mean(eval_rews)
print(f"\n  Official result -> {eval_label} checkpoint")
print(f"  Eval Mean: {eval_mean:.2f} | Eval Std: {np.std(eval_rews):.2f}")
print(f"  Phase 1 MADDPG 500K Eval Mean: -47.24")
print(f"  Q-MADDPG v3 improvement: {eval_mean-(-47.24):+.2f}")
print("=" * 65)

with open(f"{SAVE_DIR}/eval_results.csv","w",newline="") as fh:
    import csv; w=csv.writer(fh)
    w.writerow(["episode","reward","checkpoint"])
    for i,r in enumerate(eval_rews): w.writerow([i+1,round(r,4),eval_label])
print(f"Eval saved: {SAVE_DIR}/eval_results.csv")

  FIX 2: Evaluating BEST and FINAL checkpoints (100 eps each)
  [BEST  checkpoint] Mean=-45.54 Std=12.91 Max=-16.24 Min=-81.79
  [FINAL checkpoint] Mean=-46.06 Std=12.66 Max=-18.72 Min=-71.25

  Official result -> BEST checkpoint
  Eval Mean: -45.54 | Eval Std: 12.91
  Phase 1 MADDPG 500K Eval Mean: -47.24
  Q-MADDPG v3 improvement: +1.70
Eval saved: /kaggle/working/eval_results.csv


In [21]:
# CELL 20 — Training Curves + Save to Drive
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Q-MADDPG Phase 2 v3 — 500K Steps | Best Avg100={best_avg100:.2f}', fontsize=13)

ax = axes[0, 0]
ax.plot(all_ep_rewards, alpha=0.2, color='steelblue', lw=0.5)
w = 100
if len(all_ep_rewards) >= w:
    ma = np.convolve(all_ep_rewards, np.ones(w)/w, mode='valid')
    ax.plot(range(w-1, len(all_ep_rewards)), ma, color='steelblue', lw=2,
            label=f'Q-MADDPG v3 Avg100 (best={best_avg100:.1f})')
ax.axhline(best_avg100, color='green', ls='--', alpha=0.6, label=f'Best={best_avg100:.1f}')
ax.axhline(-45.71,      color='red',   ls=':',  alpha=0.7, label='Phase1 baseline=-45.71')
ax.set_title('Episode Reward'); ax.set_xlabel('Episode')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
if a_loss_hist: ax.plot(a_loss_hist, color='coral', alpha=0.7)
ax.set_title('QI Actor Loss'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
if c_loss_hist: ax.semilogy(np.maximum(c_loss_hist, 1e-6), color='mediumseagreen', alpha=0.7)
ax.set_title('Critic Loss (log)'); ax.set_xlabel('Update'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.hist(eval_rews, bins=30, color='mediumpurple', edgecolor='white')
ax.axvline(eval_mean, color='red', ls='--', label=f'Q-MADDPG Mean: {eval_mean:.1f}')
ax.axvline(-49.88,    color='orange', ls=':', label='Phase1 Mean: -49.88')
ax.set_title('Eval Reward Distribution')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_local = f'{LOCAL_DIR}/qmaddpg_v3_500K_curves.png'
plot_drive = f'{SAVE_DIR}/qmaddpg_v3_500K_curves.png'
plt.savefig(plot_local, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to Drive: {plot_drive}')

Plot saved to Drive: /kaggle/working/qmaddpg_v3_500K_curves.png


In [22]:
# CELL 21 — Final Summary + Zip All Results
print('=' * 65)
print('  Q-MADDPG Phase 2 v3 — 500K — COMPLETE')
print('=' * 65)
print(f'  Best Avg100      : {best_avg100:.2f}  (Phase1 = -45.71)')
print(f'  Eval Mean        : {np.mean(eval_rews):.2f}  (Phase1 = -47.24)')
print(f'  Eval Std         : {np.std(eval_rews):.2f}')
print(f'  Training Time    : {total_time/60:.1f} min')
print(f'  Actor Params     : {sum(p.numel() for p in actors[0].parameters())} per agent (99.4% fewer than Phase1)')
print(f'  Improvement      : {best_avg100-(-45.71):+.2f} Best Avg100 | {np.mean(eval_rews)-(-47.24):+.2f} Eval Mean')
print('=' * 65)
print(f'  All files in Drive: {SAVE_DIR}')

# Create zip
zip_local = f'{LOCAL_DIR}/QMADDPG_500K_v3.zip'
with zipfile.ZipFile(zip_local, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(LOCAL_DIR):
        for file in files:
            if file.endswith('.zip'): continue
            full = os.path.join(root, file)
            zf.write(full, os.path.relpath(full, LOCAL_DIR))
zip_drive = f'{SAVE_DIR}/QMADDPG_500K_v3.zip'
print(f'\n  Zip: {zip_drive} ({os.path.getsize(zip_drive)/1e6:.1f} MB)')
print('  Download: Kaggle Output tab -> QMAPPO_500K_v3.zip')

  Q-MADDPG Phase 2 v3 — 500K — COMPLETE
  Best Avg100      : -42.21  (Phase1 = -45.71)
  Eval Mean        : -45.54  (Phase1 = -47.24)
  Eval Std         : 12.91
  Training Time    : 207.9 min
  Actor Params     : 5413 per agent (99.4% fewer than Phase1)
  Improvement      : +3.50 Best Avg100 | +1.70 Eval Mean
  All files in Drive: /kaggle/working

  Zip: /kaggle/working/QMADDPG_500K_v3.zip (3.2 MB)
  Download: Kaggle Output tab -> QMAPPO_500K_v3.zip
